In [7]:
import sys
from pathlib import Path

# Find the project root (the folder that contains "shared")
root = Path.cwd().resolve()
while root != root.parent and not (root / "shared").exists():
    root = root.parent

if not (root / "shared").exists():
    raise RuntimeError("Could not find project root containing 'shared/'")

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print("Project root:", root)
print("sys.path[0]:", sys.path[0])

from shared.paths import get_experiment_paths
from shared.seed import seed_everything
from shared.visualization import show_grid, save_grid

Project root: D:\Projects\GANs
sys.path[0]: D:\Projects\GANs


In [8]:
EXP = "03_cdcgan"
paths = get_experiment_paths(EXP)

seed_everything(42)

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [9]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_ds = datasets.MNIST(
    root=str(paths.datasets_dir / "MNIST"),
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_ds,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from shared.weight_init import dcgan_weights_init

In [11]:
z_dim = 100
num_classes = 10
embed_dim = 100
feat = 64

class CGenerator(nn.Module):
    def __init__(self, z_dim=100, num_classes=10, embed_dim=100, img_channels=1, feat=64):
        super().__init__()
        self.embed = nn.Embedding(num_classes, embed_dim)
        in_ch = z_dim + embed_dim

        self.net = nn.Sequential(
            nn.ConvTranspose2d(in_ch, feat*4, 7, 1, 0, bias=False),   # 1 -> 7
            nn.BatchNorm2d(feat*4),
            nn.ReLU(True),

            nn.ConvTranspose2d(feat*4, feat*2, 4, 2, 1, bias=False),  # 7 -> 14
            nn.BatchNorm2d(feat*2),
            nn.ReLU(True),

            nn.ConvTranspose2d(feat*2, feat, 4, 2, 1, bias=False),    # 14 -> 28
            nn.BatchNorm2d(feat),
            nn.ReLU(True),

            nn.Conv2d(feat, img_channels, 3, 1, 1),
            nn.Tanh()
        )

    def forward(self, z, y):
        y_emb = self.embed(y).unsqueeze(2).unsqueeze(3)  # (N, embed_dim, 1, 1)
        x = torch.cat([z, y_emb], dim=1)
        return self.net(x)

class CDiscriminator(nn.Module):
    def __init__(self, num_classes=10, img_channels=1, feat=64):
        super().__init__()
        self.embed = nn.Embedding(num_classes, 28*28)

        in_ch = img_channels + 1

        self.net = nn.Sequential(
            nn.Conv2d(in_ch, feat, 4, 2, 1),                 # 28 -> 14
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feat, feat*2, 4, 2, 1, bias=False),    # 14 -> 7
            nn.BatchNorm2d(feat*2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feat*2, feat*4, 3, 2, 1, bias=False),  # 7 -> 4
            nn.BatchNorm2d(feat*4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feat*4, 1, 4, 1, 0),                   # 4 -> 1
        )

    def forward(self, x, y):
        y_map = self.embed(y).view(-1, 1, 28, 28)           # (N,1,28,28)
        inp = torch.cat([x, y_map], dim=1)                  # (N,2,28,28)
        out = self.net(inp)
        return out.view(-1)

In [ ]:
criterion = nn.BCEWithLogitsLoss()

G = CGenerator(z_dim=z_dim, num_classes=num_classes, embed_dim=embed_dim, feat=feat).to(device)
D = CDiscriminator(num_classes=num_classes, feat=feat).to(device)

G.apply(dcgan_weights_init)
D.apply(dcgan_weights_init)

g_opt = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
d_opt = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

epochs = 20

# fixed noise + fixed labels: 0..9 repeated (8 each => 80 images)
fixed_z = torch.randn(80, z_dim, 1, 1, device=device)
fixed_y = torch.arange(0, 10, device=device).repeat_interleave(8)

for epoch in range(1, epochs + 1):
    G.train(); D.train()

    for real, y in train_loader:
        real = real.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True).long()
        b = real.size(0)

        #  Train D 
        z = torch.randn(b, z_dim, 1, 1, device=device)
        fake = G(z, y).detach()

        d_real = D(real, y)
        d_fake = D(fake, y)

        real_labels = torch.full((b,), 0.9, device=device)  # label smoothing
        fake_labels = torch.zeros(b, device=device)

        d_loss = (criterion(d_real, real_labels) + criterion(d_fake, fake_labels)) / 2

        d_opt.zero_grad(set_to_none=True)
        d_loss.backward()
        d_opt.step()

        #  Train G 
        z = torch.randn(b, z_dim, 1, 1, device=device)
        fake = G(z, y)
        d_fake_for_g = D(fake, y)

        g_loss = criterion(d_fake_for_g, torch.ones(b, device=device))

        g_opt.zero_grad(set_to_none=True)
        g_loss.backward()
        g_opt.step()

    G.eval()
    with torch.no_grad():
        fixed_samples = G(fixed_z, fixed_y)

    save_grid(fixed_samples, paths.samples_dir / f"epoch_{epoch:03d}.png", nrow=8)

    torch.save({
        "epoch": epoch,
        "G": G.state_dict(),
        "D": D.state_dict(),
        "g_opt": g_opt.state_dict(),
        "d_opt": d_opt.state_dict(),
        "z_dim": z_dim,
        "feat": feat,
        "embed_dim": embed_dim,
        "num_classes": num_classes
    }, paths.models_dir / f"ckpt_epoch_{epoch:03d}.pth")

    print(f"[cDCGAN] Epoch {epoch}/{epochs} | d_loss={d_loss.item():.4f} | g_loss={g_loss.item():.4f}")

[cDCGAN] Epoch 1/20 | d_loss=0.4262 | g_loss=2.1062
[cDCGAN] Epoch 2/20 | d_loss=0.4566 | g_loss=2.1571
[cDCGAN] Epoch 3/20 | d_loss=0.2968 | g_loss=3.2620
[cDCGAN] Epoch 4/20 | d_loss=0.5828 | g_loss=3.4221
[cDCGAN] Epoch 5/20 | d_loss=0.3932 | g_loss=1.6915
[cDCGAN] Epoch 6/20 | d_loss=0.3863 | g_loss=2.6266
[cDCGAN] Epoch 7/20 | d_loss=0.3097 | g_loss=3.0904
[cDCGAN] Epoch 8/20 | d_loss=0.3211 | g_loss=2.8838
[cDCGAN] Epoch 9/20 | d_loss=0.3010 | g_loss=4.4555
[cDCGAN] Epoch 10/20 | d_loss=0.2778 | g_loss=3.3480
[cDCGAN] Epoch 11/20 | d_loss=0.3612 | g_loss=2.8845
[cDCGAN] Epoch 12/20 | d_loss=0.4463 | g_loss=4.0433
[cDCGAN] Epoch 13/20 | d_loss=0.2603 | g_loss=4.0336
[cDCGAN] Epoch 14/20 | d_loss=0.4477 | g_loss=4.8168
[cDCGAN] Epoch 15/20 | d_loss=0.2759 | g_loss=2.8686
[cDCGAN] Epoch 16/20 | d_loss=0.2978 | g_loss=2.8782
[cDCGAN] Epoch 17/20 | d_loss=0.2977 | g_loss=2.9530
[cDCGAN] Epoch 18/20 | d_loss=0.3523 | g_loss=3.1816
[cDCGAN] Epoch 19/20 | d_loss=0.3022 | g_loss=4.3130
[c